# Exploratory Data Analysis (EDA)

Summary: Exploratory analysis of the samples and age/timepoint metadata, as well as beta diversity (PCOAs).

Author, date: Fannie Kerff, January-August 2025

Environment: qiime2-amplicon-2024.10 (saved in `environment.yml`)

## Setup

In [ ]:
# imports
import pandas as pd
import matplotlib.pyplot as plt

from functions_script import blend_with_white, samples_scatter_plot, load_tsv, density_plot, generate_color_palette, plot_pcoa_infants, violinplot_with_lines, violinplot_no_lines

%matplotlib inline

In [ ]:
# paths
# IF ACCESS TO ALL STUDY DATA
path = "/cluster/work/bokulich/fkerff/GrumpyBiome-analysis"
meta_data_path = f"{path}/data-sensitive/meta_data"
figures_path = f"{path}/figures"

# ELSE (ONLY ACCESS TO PUBLIC DATA)
# meta_data_path = "../data/meta_data"
# figures_path = "../figures"

In [ ]:
# create color palettes
colors = [plt.cm.Spectral(i/float(6)) for i in range(7)]
timepoints_colors = [colors[0], colors[5], colors[6]]
lighter_timepoints_colors = [blend_with_white(c, blend_factor=0.3) for c in timepoints_colors]
pcoa_palette = generate_color_palette([10, 13, 13]) # specify number of shades per timepoint

## Load samples and infants metadata

In [ ]:
# load samples metadata
md_samples = load_tsv(f"{meta_data_path}/md_samples.tsv")
md_samples = md_samples.sort_values(by=["timepoint", "infant_id"])

print(md_samples.shape)
md_samples.head()

In [ ]:
# load infant metadata per age/timepoint
md_ages = load_tsv(f"{meta_data_path}/md_ages.tsv")
md_ages = md_ages.sort_values(by=["timepoint", "infant_id"])

print(md_ages.shape)
md_ages.head()

In [ ]:
# load all infant samples (incl. excluded) for the plot with all stool samples collection times
md_all = pd.read_csv(f"{meta_data_path}/raw/md_all.tsv", sep='\t')

In [ ]:
# load beta diversity metric tables with metadata
pcs_braycurtis_with_md = pd.read_csv(f"{meta_data_path}/pcs_braycurtis.tsv", sep='\t')
pcs_jaccard_with_md = pd.read_csv(f"{meta_data_path}/pcs_jaccard.tsv", sep='\t')
pcs_uuf_with_md = pd.read_csv(f"{meta_data_path}/pcs_uuf.tsv", sep='\t')
pcs_wuf_with_md = pd.read_csv(f"{meta_data_path}/pcs_wuf.tsv", sep='\t')

## Distribution of infants by assessment ages

In [ ]:
# total number of unique infants
md_samples.infant_id.nunique()

In [ ]:
# compute total number of samples per infant
tot_samp_per_subj = md_samples.groupby(['infant_id']).size()
print(tot_samp_per_subj.agg(['min', 'max', 'median']))

In [ ]:
# compute total number of samples per infant per timepoint/age
samp_per_subj = pd.DataFrame(md_samples.groupby(['infant_id', 'timepoint']).size())
print(samp_per_subj.agg(['min', 'max', 'median', 'std']))

##  Stool samples collection times across all ages and infants

In [ ]:
# define unique timepoints
timepoints = sorted(md_samples.timepoint.unique())

# plot and save samples scatter plot along clock time
samples_scatter_plot(
    data_analysis=md_samples,
    data_all=md_all,
    timepoints=timepoints,
    output_file=f"{figures_path}/samples_scatterplot.pdf"
)

## Density plot

In [ ]:
# plot and save density plot
density_plot(
    data_analysis=md_samples,
    timepoints=timepoints,
    colors=timepoints_colors,
    alpha=1,
    output_file=f"{figures_path}/samples_densityplot.pdf"
)

## Gut microbiota beta diversity (between-samples (dis)similarity)

In [ ]:
# variance explained from ../data/processed_data/6-diversity-3035/emperor-plots-beta-div-metrics
var_exp_bc = ["16.67%", "13.12%"]
var_exp_ja = ["9.261%", "7.052%"]
var_exp_uuf = ["17.61%", "9.714%"]
var_exp_wuf = ["31.32%", "23.88%"]

# plot and save PCoA plots
fig, axs = plt.subplots(2, 2, figsize=(12, 12))

plot_pcoa_infants(pcs_braycurtis_with_md, 'infant_id', "Bray-Curtis dissimilarity", axs[0, 0], var_exp_bc)
plot_pcoa_infants(pcs_jaccard_with_md, 'infant_id', "Jaccard similarity index", axs[0, 1], var_exp_ja)
plot_pcoa_infants(pcs_uuf_with_md, 'infant_id',"Unweighted UniFrac", axs[1, 0], var_exp_uuf)
plot_pcoa_infants(pcs_wuf_with_md, 'infant_id', "Weighted UniFrac", axs[1, 1], var_exp_wuf)

plt.tight_layout()
plt.savefig(f"{figures_path}/pcoas_subjects.pdf", dpi=300, bbox_inches='tight')
plt.show()

## Infant sleep BISQ variables across all ages: sleep duration, sleep onset latency, bedtime and frequency of waking in the night variables.

In [ ]:
# drop rows with NaN in babySQUID
data_BISQ = md_ages.dropna(subset=['babySQUID'])

# plot and save
fig, axs = plt.subplots(1, 4, figsize=(15, 6))

violinplot_with_lines(data_BISQ, "timepoint", "nighttime_sleep_duration_in_h", "infant_id", "", "Nighttime  sleep duration (h)", lighter_timepoints_colors, axs[0], fontsize_title=14)
violinplot_with_lines(data_BISQ, "timepoint", "sleep_latency_in_h", "infant_id", "", "Sleep onset latency (h)", lighter_timepoints_colors, axs[1], fontsize_title=14)
violinplot_with_lines(data_BISQ, "timepoint", "bedtime_in_h", "infant_id", "", "Bedtime (h)", lighter_timepoints_colors, axs[2], fontsize_title=14)
violinplot_with_lines(data_BISQ, "timepoint", "number_of_awakenings", "infant_id", "", "Number of nighttime awakenings", lighter_timepoints_colors, axs[3], fontsize_title=14)

plt.tight_layout()
plt.savefig(f"{figures_path}/BISQ.pdf", dpi=300, bbox_inches='tight')
plt.show()

## BCQ variables across all ages: structured and attuned care scores

In [ ]:
# drop rows with NaN in BCQ_Structure
data_BCQ = md_ages.dropna(subset=['BCQ_Structure'])

# plot and save
fig, axs = plt.subplots(1, 2, figsize=(15, 6))

violinplot_with_lines(data_BCQ, "timepoint", "BCQ_Structure", "infant_id", "", "Parenting style: BCQ structure score", lighter_timepoints_colors, axs[0])
violinplot_with_lines(data_BCQ, "timepoint", "BCQ_Attunement", "infant_id", "", "Parenting style: BCQ attunement score", lighter_timepoints_colors, axs[1])

plt.savefig(f"{figures_path}/BCQ.pdf", dpi=300, bbox_inches='tight')
plt.show()

## ASQ Composite score across all ages: represents communication, fine motor, gross motor, personal-social and problem solving scores.

In [ ]:
# drop rows with NaN in ASQ_Composite
data_ASQ = md_ages.dropna(subset=['ASQ_Composite'])

# plot and save
fig, axs = plt.subplots(1, 1, figsize=(6, 6))

violinplot_with_lines(data_ASQ, "timepoint", "ASQ_Composite", "infant_id", "", "ASQ: Composite score", lighter_timepoints_colors, axs)

plt.savefig(f"{figures_path}/ASQ_composite.pdf", dpi=300, bbox_inches='tight')
plt.show()

## Clock times of stool samples and sleep and feeding history at each stool sample time

In [ ]:
# drop rows with NaN in directly preceding feeding and sleeping information:
data_sleep = md_samples.dropna(subset=['prior_time_spent_awake_in_h'])
data_sleep_dur = md_samples.dropna(subset=['duration_of_last_sleep_in_h'])
data_feeding = md_samples.dropna(subset=['time_since_last_feeding_in_h'])

# plot and save
fig, axs = plt.subplots(1, 4, figsize=(15, 6))

violinplot_no_lines(md_samples, "timepoint", "hour_stool_sample", "", "Clocktime of the stool sample (h)", lighter_timepoints_colors, axs[0], fontsize_title=12)
violinplot_no_lines(data_sleep, "timepoint", "prior_time_spent_awake_in_h", "", "Prior time spent awake (h)", lighter_timepoints_colors, axs[1], fontsize_title=12)
violinplot_no_lines(data_sleep_dur, "timepoint", "duration_of_last_sleep_in_h", "", "Last sleep duration (h)", lighter_timepoints_colors, axs[2], fontsize_title=12)
violinplot_no_lines(data_feeding, "timepoint", "time_since_last_feeding_in_h", "", "Time since the last feeding (h)", lighter_timepoints_colors, axs[3], fontsize_title=12)

plt.tight_layout()
plt.savefig(f"{figures_path}/samples_EDA.pdf", dpi=300, bbox_inches='tight')
plt.show()

## Gut melatonin and time since the last bowel movement at each stool sample

In [ ]:
# drop rows with NaN in Melatonin in Stool and time_since_last_bowel_movement_in_h
data_mel = md_samples.dropna(subset=['Melatonin in Stool (pg/g)'])
data_bowel = md_samples.dropna(subset=['time_since_last_bowel_movement_in_h'])

# plot and save
fig, axs = plt.subplots(1, 2, figsize=(15, 6))

violinplot_no_lines(data_mel, "timepoint", "Melatonin in Stool (pg/g)", "", "Melatonin in stool (pg/g)", lighter_timepoints_colors, axs[0], fontsize_title=12)
violinplot_no_lines(data_bowel, "timepoint", "time_since_last_bowel_movement_in_h", "", "Time since the last bowel movement (h)", lighter_timepoints_colors, axs[1], fontsize_title=12)

plt.tight_layout()
plt.savefig(f"{figures_path}/melatonin_EDA.pdf", dpi=300, bbox_inches='tight')
plt.show()

## Summarizing metadata (Table 1)

In [ ]:
# number of unique infants per timepoint
md_samples.groupby("timepoint")["infant_id"].nunique()

In [ ]:
# compute total number of twins per timepoint (1 => no twins)
md_samples[md_samples["twin"] == "yes"].groupby("timepoint")["infant_id"].nunique()

In [ ]:
# print summary statistics of age_days per timepoint
md_samples.groupby("timepoint")["age_days"].describe()

In [ ]:
# print summary statistics of hour_stool_sample per timepoint
md_samples.groupby("timepoint")["hour_stool_sample"].describe()

In [ ]:
# print summary statistics of observed_features per timepoint
md_samples.groupby("timepoint")["observed_features"].describe()

In [ ]:
# print summary statistics of shannon_entropy per timepoint
md_samples.groupby("timepoint")["shannon_entropy"].describe()

In [ ]:
# print summary statistics of pielou_evenness per timepoint
md_samples.groupby("timepoint")["pielou_evenness"].describe()

In [ ]:
# print summary statistics of faith_pd per timepoint
md_samples.groupby("timepoint")["faith_pd"].describe()

In [ ]:
# print summary statistics of the cosine fit of observed features
md_ages.groupby("timepoint")["R2_abundance"].describe()

In [ ]:
# print summary statistics of the cosine fit of Shannon entropy 
md_ages.groupby("timepoint")["R2_abundance_evenness"].describe()

In [ ]:
# print summary statistics of the cosine fit of Pielou evenness
md_ages.groupby("timepoint")["R2_evenness"].describe()

In [ ]:
# print summary statistics of the cosine fit of Faith phylogenetic diversity 
md_ages.groupby("timepoint")["R2_biodiversity"].describe()

In [ ]:
# print summary statistics of the cosine fit of veillonella relative abundance
md_ages.groupby("timepoint")["R2_veillo"].describe()

In [ ]:
# print summary statistics of the cosine fit of bifidobacterium relative abundance
md_ages.groupby("timepoint")["R2_bifido"].describe()

In [ ]:
# print summary statistics of the cosine fit of escherichia relative abundance
md_ages.groupby("timepoint")["R2_esche"].describe()

In [ ]:
# print summary statistics of the cosine fit of bacteroides relative abundance
md_ages.groupby("timepoint")["R2_bacteroides"].describe()

In [ ]:
# print summary statistics of the cosine fit of clostridium relative abundance
md_ages.groupby("timepoint")["R2_clostridium"].describe()

In [ ]:
# print summary statistics of microbial temporal volatility based on Bray-Curtis dissimilarity 
md_ages.groupby("timepoint")["volatility_braycurtis"].describe()

In [ ]:
# print summary statistics of microbial temporal volatility based on Jaccard similarity index 
md_ages.groupby("timepoint")["volatility_jaccard"].describe()

In [ ]:
# print summary statistics of microbial temporal volatility based on Unweighted UniFrac distance 
md_ages.groupby("timepoint")["volatility_unweighted_unifrac"].describe()

In [ ]:
# print summary statistics of microbial temporal volatility based on Weighted UniFrac distance 
md_ages.groupby("timepoint")["volatility_weighted_unifrac"].describe()

In [ ]:
# print summary statistics of the babySQUID per timepoint
md_ages.groupby("timepoint")["babySQUID"].describe()

In [ ]:
# print summary statistics of nighttime_sleep_duration_in_h per timepoint
md_ages.groupby("timepoint")["nighttime_sleep_duration_in_h"].describe()

In [ ]:
# print summary statistics of sleep_latency_in_h per timepoint
md_ages.groupby("timepoint")["sleep_latency_in_h"].describe()

In [ ]:
# print summary statistics of bedtime_in_h per timepoint
md_ages.groupby("timepoint")["bedtime_in_h"].describe()

In [ ]:
# print summary statistics of number_of_awakenings per timepoint
md_ages.groupby("timepoint")["number_of_awakenings"].describe()

In [ ]:
# print summary statistics of the CFI per timepoint
md_ages.groupby("timepoint")["CFI"].describe()

In [ ]:
# print summary statistics of BCQ_Attunement per timepoint
md_ages.groupby("timepoint")["BCQ_Attunement"].describe()

In [ ]:
# print summary statistics of ASQ_Composite per timepoint
md_ages.groupby("timepoint")["ASQ_Composite"].describe()

In [ ]:
# print summary statistics of prior_time_spent_awake_in_h per timepoint
md_samples.groupby("timepoint")["prior_time_spent_awake_in_h"].describe()

In [ ]:
# print summary statistics of duration_of_last_sleep_in_h per timepoint
md_samples.groupby("timepoint")["duration_of_last_sleep_in_h"].describe()

In [ ]:
# print summary statistics of time_since_last_feeding_in_h per timepoint
md_samples.groupby("timepoint")["time_since_last_feeding_in_h"].describe()

In [ ]:
# print summary statistics of Melatonin in Stool (pg/g) per timepoint
md_samples.groupby("timepoint")["Melatonin in Stool (pg/g)"].describe()